# Equity Data Pipeline — Validation Notebook

This notebook connects to your Cloudflare R2 bucket using DuckDB to check schemas, row counts, data cleaning rules, and run basic return analytics queries.

In [ ]:
import os
import duckdb
from dotenv import load_dotenv

# Search up to 3 levels of parent directories to find the .env file
dotenv_path = None
for i in range(3):
    candidate = os.path.join('../' * i, '.env')
    if os.path.exists(candidate):
        dotenv_path = candidate
        break

if dotenv_path:
    load_dotenv(dotenv_path)
else:
    load_dotenv()

account_id = os.environ.get('R2_ACCOUNT_ID')
access_key_id = os.environ.get('R2_ACCESS_KEY_ID')
secret_access_key = os.environ.get('R2_SECRET_ACCESS_KEY')
bucket_name = os.environ.get('R2_BUCKET_NAME', 'equity-data-lake')
r2_endpoint = f'{account_id}.r2.cloudflarestorage.com'

# Connect to local in-memory DuckDB
con = duckdb.connect()
con.execute('INSTALL httpfs; LOAD httpfs;')
con.execute(f'''
    SET s3_endpoint = "{r2_endpoint}";
    SET s3_access_key_id = "{access_key_id}";
    SET s3_secret_access_key = "{secret_access_key}";
    SET s3_region = "auto";
''')
print('DuckDB successfully loaded and authenticated with R2.')

## 1. Table Definitions and R2 Paths

In [ ]:
tables = {
    'dim_company': f's3://{bucket_name}/silver/dim_company/dim_company.parquet',
    'fact_daily_prices': f's3://{bucket_name}/silver/fact_daily_prices/**/*.parquet',
    'fact_income_statement': f's3://{bucket_name}/silver/fact_income_statement/fact_income_statement.parquet',
    'fact_balance_sheet': f's3://{bucket_name}/silver/fact_balance_sheet/fact_balance_sheet.parquet',
    'fact_cash_flow': f's3://{bucket_name}/silver/fact_cash_flow/fact_cash_flow.parquet',
    'fact_earnings': f's3://{bucket_name}/silver/fact_earnings/fact_earnings.parquet',
    'fact_dividends': f's3://{bucket_name}/silver/fact_dividends/fact_dividends.parquet',
    'fact_splits': f's3://{bucket_name}/silver/fact_splits/fact_splits.parquet'
}

## 2. Verify Row Counts and Types

We query all silver Parquet tables in R2 to check counts, check schemas, and verify that there are zero literal strings of 'None' remaining.

In [ ]:
for name, path in tables.items():
    try:
        count = con.execute(f"SELECT COUNT(*) FROM read_parquet('{path}')").fetchone()[0]
        print(f'Table: {name:<25} | Row Count: {count}')
        
        # Check for any literal string 'None' values in text columns
        columns_df = con.execute(f"DESCRIBE SELECT * FROM read_parquet('{path}')").df()
        char_cols = columns_df[columns_df['column_type'].str.contains('VARCHAR', na=False)]['column_name'].tolist()
        if char_cols:
            sum_exprs = [f"SUM(CASE WHEN {col} = 'None' THEN 1 ELSE 0 END) AS count_{col}" for col in char_cols]
            none_query = f"SELECT {', '.join(sum_exprs)} FROM read_parquet('{path}')"
            none_counts = con.execute(none_query).df().sum().sum()
            if none_counts > 0:
                print(f'  [WARNING] Literal "None" values found in character columns!')
            else:
                print(f'  [OK] Zero literal "None" strings found in columns: {char_cols}')
    except Exception as e:
        print(f'  [ERROR] Failed to query {name}: {e}')

## 3. Verify point-in-time universe queries

In [ ]:
df_company = con.execute(f'''
    SELECT symbol, name, exchange, asset_type, listing_status, ipo_date, delisted_date
    FROM read_parquet('{tables['dim_company']}')
    ORDER BY symbol
''').df()
df_company

## 4. Verify return calculation analytical queries

This query joins fact_daily_prices to compute stock returns over the available trading history.

In [ ]:
df_returns = con.execute(f'''
    SELECT 
        symbol, 
        MIN(trade_date) as start_date, 
        MAX(trade_date) as end_date,
        COUNT(*) as trading_days,
        FIRST(adjusted_close ORDER BY trade_date ASC) as start_price,
        LAST(adjusted_close ORDER BY trade_date ASC) as end_price,
        ROUND((end_price - start_price) / start_price * 100, 2) as return_pct
    FROM read_parquet('{tables['fact_daily_prices']}')
    GROUP BY symbol
    ORDER BY return_pct DESC
''').df()
df_returns